# Validate LLM Ground Truth Against Human Labels

This notebook is an audit-only validation notebook. It does not create training data, does not modify retrieval runs, and does not expose or write stable qrels.

Its only purpose is to compare completed human relevance labels against the LLM-generated ground truth and print agreement metrics such as quadratic weighted Cohen's kappa, exact accuracy, binary accuracy, and a confusion matrix.


## 1. Setup

Expected inputs:

- `groundtruth_outputs/annotation/llm_groundtruth_labels.jsonl`
- `groundtruth_outputs/human_validation/human_validation_annotation_sheet_completed.csv`

The human-completed CSV must contain at least:

- `query_id`
- `doc_id`
- `human_relevance`

The LLM labels are treated as the predicted labels. The human labels are treated as the audit reference labels.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import pandas as pd


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
HUMAN_VALIDATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "human_validation"

LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
HUMAN_COMPLETED_LABELS_PATH = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet_completed.csv"

RELEVANCE_LABELS = [0, 1, 2, 3]
BINARY_RELEVANCE_THRESHOLD = 1

print("Finalproject root:", FINALPROJECT_ROOT)
print("LLM labels path:", LLM_LABELS_PATH)
print("Human completed labels path:", HUMAN_COMPLETED_LABELS_PATH)


## 2. Load Labels

This section only loads existing files. It does not write any validation artifacts. If the completed human file is missing, create it outside this notebook from the blinded annotation items and fill `human_relevance` manually.


In [ ]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {jsonl_path}:{line_number}") from error
    return records


def load_llm_labels(llm_labels_path: Path) -> pd.DataFrame:
    """Load LLM labels as one row per query-document pair."""
    if not llm_labels_path.exists():
        raise FileNotFoundError(f"LLM labels file not found: {llm_labels_path}")

    label_dataframe = pd.DataFrame(load_jsonl_records(llm_labels_path))
    required_columns = {"query_id", "doc_id", "relevance"}
    missing_columns = required_columns - set(label_dataframe.columns)
    if missing_columns:
        raise ValueError(f"LLM labels are missing required columns: {sorted(missing_columns)}")

    label_dataframe = label_dataframe.rename(columns={"relevance": "llm_relevance"}).copy()
    label_dataframe["query_id"] = label_dataframe["query_id"].astype(int)
    label_dataframe["doc_id"] = label_dataframe["doc_id"].astype(int)
    label_dataframe["llm_relevance"] = label_dataframe["llm_relevance"].astype(int)

    duplicate_count = label_dataframe.duplicated(subset=["query_id", "doc_id"]).sum()
    if duplicate_count:
        raise ValueError(f"LLM labels contain {duplicate_count} duplicate query-doc pairs.")

    return label_dataframe


def load_human_completed_labels(human_completed_path: Path) -> pd.DataFrame:
    """Load completed human labels."""
    if not human_completed_path.exists():
        raise FileNotFoundError(
            "Completed human validation file not found. Expected path: "
            f"{human_completed_path}"
        )

    human_dataframe = pd.read_csv(human_completed_path)
    required_columns = {"query_id", "doc_id", "human_relevance"}
    missing_columns = required_columns - set(human_dataframe.columns)
    if missing_columns:
        raise ValueError(f"Human labels are missing required columns: {sorted(missing_columns)}")

    human_dataframe = human_dataframe.dropna(subset=["human_relevance"]).copy()
    human_dataframe["query_id"] = human_dataframe["query_id"].astype(int)
    human_dataframe["doc_id"] = human_dataframe["doc_id"].astype(int)
    human_dataframe["human_relevance"] = human_dataframe["human_relevance"].astype(int)

    invalid_labels = sorted(set(human_dataframe["human_relevance"]) - set(RELEVANCE_LABELS))
    if invalid_labels:
        raise ValueError(f"Human labels contain invalid relevance values: {invalid_labels}")

    duplicate_count = human_dataframe.duplicated(subset=["query_id", "doc_id"]).sum()
    if duplicate_count:
        raise ValueError(f"Human labels contain {duplicate_count} duplicate query-doc pairs.")

    return human_dataframe


def build_validation_comparison(
    llm_label_dataframe: pd.DataFrame,
    human_label_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Merge LLM and human labels on query_id/doc_id."""
    comparison_dataframe = human_label_dataframe.merge(
        llm_label_dataframe[["query_id", "doc_id", "llm_relevance"]],
        on=["query_id", "doc_id"],
        how="inner",
    )

    if comparison_dataframe.empty:
        raise ValueError("No overlapping query-doc pairs between human labels and LLM labels.")

    missing_llm_count = len(human_label_dataframe) - len(comparison_dataframe)
    if missing_llm_count:
        print(f"Warning: {missing_llm_count} human-labeled rows do not have matching LLM labels.")

    return comparison_dataframe


llm_label_dataframe = load_llm_labels(LLM_LABELS_PATH)
print("LLM label rows:", len(llm_label_dataframe))
print("LLM queries:", llm_label_dataframe["query_id"].nunique())
print("LLM labels per query:")
print(llm_label_dataframe.groupby("query_id").size().describe())

if HUMAN_COMPLETED_LABELS_PATH.exists():
    human_label_dataframe = load_human_completed_labels(HUMAN_COMPLETED_LABELS_PATH)
    comparison_dataframe = build_validation_comparison(llm_label_dataframe, human_label_dataframe)
    print("\nHuman label rows:", len(human_label_dataframe))
    print("Human queries:", human_label_dataframe["query_id"].nunique())
    print("Compared rows:", len(comparison_dataframe))
else:
    human_label_dataframe = pd.DataFrame()
    comparison_dataframe = pd.DataFrame()
    print("\nCompleted human validation file does not exist yet.")
    print("Create/fill this file, then rerun the metric cells:")
    print(HUMAN_COMPLETED_LABELS_PATH)


## 3. Agreement Metrics

The metrics are computed in memory and printed only.

- Exact accuracy checks whether the 0-3 labels match exactly.
- Within-one accuracy allows small disagreements such as `2` vs `3`.
- Binary accuracy collapses labels into non-relevant (`0`) and relevant (`1`, `2`, `3`).
- Quadratic weighted Cohen's kappa measures ordinal agreement and penalizes large disagreements more strongly.


In [ ]:
def build_confusion_matrix(
    human_labels: list[int],
    llm_labels: list[int],
    labels: list[int],
) -> pd.DataFrame:
    """Build a confusion matrix with human labels as rows and LLM labels as columns."""
    label_to_index = {label: index for index, label in enumerate(labels)}
    matrix = [[0 for _ in labels] for _ in labels]
    for human_label, llm_label in zip(human_labels, llm_labels):
        matrix[label_to_index[int(human_label)]][label_to_index[int(llm_label)]] += 1
    return pd.DataFrame(
        matrix,
        index=[f"human_{label}" for label in labels],
        columns=[f"llm_{label}" for label in labels],
    )


def quadratic_weighted_kappa(
    human_labels: list[int],
    llm_labels: list[int],
    labels: list[int],
) -> float:
    """Compute quadratic weighted Cohen's kappa without requiring sklearn."""
    if len(human_labels) != len(llm_labels):
        raise ValueError("human_labels and llm_labels must have the same length.")
    if not human_labels:
        return 0.0

    label_count = len(labels)
    label_to_index = {label: index for index, label in enumerate(labels)}

    observed_matrix = [[0.0 for _ in labels] for _ in labels]
    human_histogram = [0.0 for _ in labels]
    llm_histogram = [0.0 for _ in labels]

    for human_label, llm_label in zip(human_labels, llm_labels):
        human_index = label_to_index[int(human_label)]
        llm_index = label_to_index[int(llm_label)]
        observed_matrix[human_index][llm_index] += 1.0
        human_histogram[human_index] += 1.0
        llm_histogram[llm_index] += 1.0

    total_count = float(len(human_labels))
    expected_matrix = [
        [(human_histogram[i] * llm_histogram[j]) / total_count for j in range(label_count)]
        for i in range(label_count)
    ]

    weighted_observed = 0.0
    weighted_expected = 0.0
    max_distance_squared = float((label_count - 1) ** 2)
    for i in range(label_count):
        for j in range(label_count):
            weight = ((i - j) ** 2) / max_distance_squared
            weighted_observed += weight * observed_matrix[i][j]
            weighted_expected += weight * expected_matrix[i][j]

    if weighted_expected == 0:
        return 1.0 if weighted_observed == 0 else 0.0
    return 1.0 - (weighted_observed / weighted_expected)


def compute_agreement_metrics(comparison_dataframe: pd.DataFrame) -> dict:
    """Compute agreement metrics between human labels and LLM labels."""
    if comparison_dataframe.empty:
        raise ValueError("comparison_dataframe is empty. Load completed human labels first.")

    human_labels = comparison_dataframe["human_relevance"].astype(int).tolist()
    llm_labels = comparison_dataframe["llm_relevance"].astype(int).tolist()
    absolute_errors = (comparison_dataframe["human_relevance"] - comparison_dataframe["llm_relevance"]).abs()

    human_binary = (comparison_dataframe["human_relevance"] >= BINARY_RELEVANCE_THRESHOLD).astype(int)
    llm_binary = (comparison_dataframe["llm_relevance"] >= BINARY_RELEVANCE_THRESHOLD).astype(int)

    metrics = {
        "compared_pairs": int(len(comparison_dataframe)),
        "compared_queries": int(comparison_dataframe["query_id"].nunique()),
        "exact_accuracy": float((absolute_errors == 0).mean()),
        "within_one_accuracy": float((absolute_errors <= 1).mean()),
        "binary_accuracy": float((human_binary == llm_binary).mean()),
        "mean_absolute_label_error": float(absolute_errors.mean()),
        "quadratic_weighted_kappa": float(
            quadratic_weighted_kappa(human_labels, llm_labels, labels=RELEVANCE_LABELS)
        ),
        "severe_0_vs_3_disagreement_count": int(
            (
                ((comparison_dataframe["human_relevance"] == 0) & (comparison_dataframe["llm_relevance"] == 3))
                | ((comparison_dataframe["human_relevance"] == 3) & (comparison_dataframe["llm_relevance"] == 0))
            ).sum()
        ),
    }
    return metrics


def print_agreement_report(comparison_dataframe: pd.DataFrame) -> dict:
    """Print agreement metrics and return them as a dictionary."""
    metrics = compute_agreement_metrics(comparison_dataframe)
    confusion_matrix = build_confusion_matrix(
        human_labels=comparison_dataframe["human_relevance"].astype(int).tolist(),
        llm_labels=comparison_dataframe["llm_relevance"].astype(int).tolist(),
        labels=RELEVANCE_LABELS,
    )

    print("Agreement metrics")
    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"{metric_name}: {metric_value:.4f}")
        else:
            print(f"{metric_name}: {metric_value}")

    print("\nHuman label distribution:")
    print(comparison_dataframe["human_relevance"].value_counts().sort_index())

    print("\nLLM label distribution on compared rows:")
    print(comparison_dataframe["llm_relevance"].value_counts().sort_index())

    print("\nConfusion matrix:")
    display(confusion_matrix)

    return metrics


if not comparison_dataframe.empty:
    agreement_metrics = print_agreement_report(comparison_dataframe)
else:
    print("No agreement metrics computed because the completed human validation file is missing.")


## 4. Disagreement Inspection

This final section prints the largest disagreements for manual inspection. It is diagnostic only and does not save any files.


In [ ]:
def show_largest_disagreements(comparison_dataframe: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    """Return rows with the largest absolute disagreement."""
    if comparison_dataframe.empty:
        raise ValueError("comparison_dataframe is empty. Load completed human labels first.")

    inspection_dataframe = comparison_dataframe.copy()
    inspection_dataframe["absolute_error"] = (
        inspection_dataframe["human_relevance"] - inspection_dataframe["llm_relevance"]
    ).abs()
    sorted_dataframe = inspection_dataframe.sort_values(
        ["absolute_error", "query_id", "doc_id"],
        ascending=[False, True, True],
    ).head(top_n)
    return sorted_dataframe


if not comparison_dataframe.empty:
    display(show_largest_disagreements(comparison_dataframe, top_n=30))
else:
    print("No disagreement inspection available because the completed human validation file is missing.")
